# Week 2 Homework: Controlled Improvement and Error Analysis

**ECBS5200 — Practical Deep Learning Engineering**

In the lab you analyzed 4 pre-trained models and diagnosed what produced
each one. Now you train your own. This notebook has you run experiments
on the **full dataset**, implement early stopping, do your first error
analysis, and discover how noisy your validation metrics actually are.

**Expected time:** ~5-6 hours on a Kaggle T4 GPU.
Training runs take ~30-40 min each. Use that time to plan your memo.

**How to use this notebook:**
- **GUIDED** cells run as-is. Read them.
- **INTERACTIVE** cells require you to fill in values or write answers.
- `___` is a placeholder that will cause a NameError if not filled in.
- The last section is your memo draft.

## Kaggle setup

1. **Persistence** → "Variables and Files"
2. **Internet** → On
3. **Accelerator** → GPU T4

In [ ]:
import subprocess, sys, os

# HuggingFace config — set BEFORE importing HF libraries
if os.path.exists('/kaggle/working'):
    os.environ['HF_HOME'] = '/kaggle/working/.hf_cache'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.53", "datasets", "accelerate", "scikit-learn",
    "matplotlib", "pandas", "tqdm",
])
print("Packages installed.")

from huggingface_hub import login
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace.")
else:
    print("WARNING: No HF_TOKEN found.")

In [ ]:
import gc
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import LambdaLR
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

REPO_PATH = Path("../..").resolve()
if not (REPO_PATH / "utils" / "data_utils.py").exists():
    REPO_PATH = Path("/tmp/course")
    if not REPO_PATH.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/earino/applied-deep-learning.git", str(REPO_PATH)])
sys.path.insert(0, str(REPO_PATH))

from utils.data_utils import (
    load_course_data, LABEL_LIST, NUM_LABELS, LABEL_TO_ID, ID_TO_LABEL, MAX_LENGTH,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "answerdotai/ModernBERT-base"

if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    AMP_DTYPE = torch.bfloat16 if cc[0] >= 8 else torch.float16
else:
    AMP_DTYPE = torch.float32

use_scaler = (AMP_DTYPE == torch.float16)
SAVE_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

print(f"Device: {device} | AMP: {AMP_DTYPE}")

---
## Part 1: Setup and Data (~10 min)
---

In [ ]:
# GUIDED: Load full dataset (no subset — this is the homework)
print("Loading and tokenizing full dataset...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds, val_ds, _ = load_course_data(tokenizer=tokenizer, max_length=MAX_LENGTH)
print(f"  Train: {len(train_ds):,}  Val: {len(val_ds):,} ({time.time()-t0:.1f}s)")


class ComplaintDataset(Dataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["label"], dtype=torch.long),
        }

### Class weighting: understanding the formula

In the lab you saw that one model used class weighting. Before running experiments,
let's understand what we're doing and compare it to sklearn's built-in
`compute_class_weight("balanced")`.

In [ ]:
# GUIDED: Count training examples per class
full_labels = np.array(train_ds["label"])
counts = np.bincount(full_labels, minlength=NUM_LABELS).astype(np.float32)
counts = np.maximum(counts, 1.0)  # avoid division by zero

print(f"Largest class:  {int(counts.max())} examples")
print(f"Smallest class: {int(counts.min())} examples")
print(f"Ratio:          {counts.max() / counts.min():.0f}x")

In [ ]:
# INTERACTIVE: Compute sqrt-inverse class weights.
# The formula: weight = sqrt(max_count / count), then normalize to mean=1.
# You saw this in the lecture and its effect in the lab (Model Alpha).
# Now implement it yourself.

sqrt_inv_weights = ___                    # sqrt of (max count / each count)
sqrt_inv_weights = ___ / ___              # normalize so the mean equals 1.0

print(f"Weights: min={sqrt_inv_weights.min():.2f}, max={sqrt_inv_weights.max():.2f}, "
      f"mean={sqrt_inv_weights.mean():.2f}")
assert abs(sqrt_inv_weights.mean() - 1.0) < 0.01, "Weights should be normalized to mean=1"

In [ ]:
# INTERACTIVE: Create the weighted loss function.
# Convert your weights to a torch tensor on the correct device,
# then pass them to nn.CrossEntropyLoss.

class_weight_tensor = ___   # torch.tensor(..., dtype=torch.float32).to(device)
weighted_loss = ___          # nn.CrossEntropyLoss(weight=...)

print(f"Weighted loss function ready. Weight tensor shape: {class_weight_tensor.shape}")

### How does this compare to sklearn's approach?

In [ ]:
# GUIDED: sklearn's compute_class_weight("balanced") for comparison
from sklearn.utils.class_weight import compute_class_weight

unique_classes = np.unique(full_labels)
balanced_raw = compute_class_weight("balanced", classes=unique_classes, y=full_labels)
balanced = np.ones(NUM_LABELS, dtype=np.float32)
for cid, wt in zip(unique_classes, balanced_raw):
    balanced[cid] = wt

print("Weight comparison (rarest and most common classes):")
print(f"{'Class':<50} {'Count':>6} {'Sqrt-inv':>9} {'Balanced':>9}")
print("-" * 80)
sorted_by_count = sorted(Counter(full_labels).items(), key=lambda x: x[1])
for label_id, count in sorted_by_count[:5] + sorted_by_count[-3:]:
    name = LABEL_LIST[label_id][:48]
    print(f"{name:<50} {count:>6} {sqrt_inv_weights[label_id]:>9.2f} {balanced[label_id]:>9.2f}")

print(f"\nSqrt-inverse: max weight = {sqrt_inv_weights.max():.1f}x")
print(f"Balanced:     max weight = {balanced.max():.0f}x")
print(f"\nBalanced is {balanced.max() / sqrt_inv_weights.max():.0f}x more aggressive.")

---
## Part 2: Experiments (~2.5 hours, mostly training)

Each experiment uses this template. Fill it in before and after each run.

```
EXPERIMENT: [name]
──────────────────
Variable changed:  [exactly ONE thing]
Held constant:     [list everything else]
Prediction:        [what you expect and why — write this BEFORE running]
Result:            [what actually happened — fill in AFTER running]
Conclusion:        [what you learned]
Meaningful?        [is the difference large enough to trust?]
```
---

In [ ]:
# Shared training function
NUM_EPOCHS = 3


def run_experiment(name, batch_size=32, lr=2e-5, loss_fn=None, num_epochs=NUM_EPOCHS):
    """Train from scratch on full data, evaluate on val. Returns metrics + predictions."""
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {name}")
    print(f"  bs={batch_size}, lr={lr}, epochs={num_epochs}")
    print(f"{'='*60}")

    t_loader = DataLoader(ComplaintDataset(train_ds), batch_size=batch_size, shuffle=True)
    v_loader = DataLoader(ComplaintDataset(val_ds), batch_size=64, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, attn_implementation="sdpa",
        id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
    ).to(device)

    if loss_fn is None:
        loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(t_loader) * num_epochs
    warmup_steps = max(1, total_steps // 10)

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    scheduler = LambdaLR(optimizer, lr_lambda)

    scaler = GradScaler("cuda", enabled=use_scaler)

    wall_start = time.time()
    epoch_history = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss, n = 0.0, 0
        progress = tqdm(t_loader, desc=f"  Epoch {epoch}/{num_epochs}", leave=True)
        for batch in progress:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            with autocast("cuda", dtype=AMP_DTYPE, enabled=(device.type == "cuda")):
                logits = model(input_ids=ids, attention_mask=mask).logits
                loss = loss_fn(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item() * labels.size(0)
            n += labels.size(0)
            progress.set_postfix(loss=f"{running_loss/n:.4f}")

        # Per-epoch validation
        model.eval()
        ep_preds, ep_labels = [], []
        with torch.no_grad():
            for batch in v_loader:
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                with autocast("cuda", dtype=AMP_DTYPE, enabled=(device.type == "cuda")):
                    logits = model(input_ids=ids, attention_mask=mask).logits
                ep_preds.extend(logits.argmax(dim=-1).cpu().numpy())
                ep_labels.extend(batch["labels"].cpu().numpy())

        ep_acc = accuracy_score(ep_labels, ep_preds)
        ep_f1 = f1_score(ep_labels, ep_preds, average="macro", zero_division=0)
        epoch_history.append({"epoch": epoch, "train_loss": running_loss / n,
                              "val_acc": ep_acc, "val_macro_f1": ep_f1})
        print(f"  Epoch {epoch}: loss={running_loss/n:.4f}, acc={ep_acc:.4f}, f1={ep_f1:.4f}")

    train_time = time.time() - wall_start

    # Final eval
    final = epoch_history[-1]
    preds = np.array(ep_preds)
    labels = np.array(ep_labels)
    zero_f1 = sum(1 for c in range(NUM_LABELS)
                  if f1_score(labels == c, preds == c, zero_division=0) == 0.0
                  and np.sum(labels == c) > 0)

    print(f"\n  Final: acc={final['val_acc']:.4f}, f1={final['val_macro_f1']:.4f}, "
          f"zero_f1={zero_f1}, time={train_time:.0f}s")

    del model, optimizer, scaler, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "name": name,
        "accuracy": round(final["val_acc"], 4),
        "macro_f1": round(final["val_macro_f1"], 4),
        "zero_f1_classes": zero_f1,
        "train_time_sec": round(train_time, 1),
        "history": epoch_history,
        "val_preds": preds,
        "val_labels": labels,
    }

### Experiment 1: Sqrt-inverse class weighting on full data (~32 min)

In the lab you saw what class weighting does to a model's metrics. Now
train your own class-weighted model on the full 58K training set.

**Fill in your experiment template BEFORE running:**

```
EXPERIMENT: Sqrt-inverse class weighting, full data
──────────────────────────────────────────────────
Variable changed:  ___
Held constant:     ___
Prediction:        ___
```

YOUR PREDICTION:



In [ ]:
# GUIDED: Run experiment 1 (uses the weighted_loss you created above)
exp1 = run_experiment(
    name="sqrt-inverse bs=32",
    batch_size=32,
    lr=2e-5,
    loss_fn=weighted_loss,
)

# Save incrementally
pd.DataFrame([{k: v for k, v in exp1.items() if k not in ("history", "val_preds", "val_labels")}]).to_csv(
    SAVE_DIR / "week2_hw_results.csv", index=False)

**Complete the experiment template with your results:**

```
Result:      ___
Conclusion:  ___
Meaningful?  ___
```

YOUR RESULT:



### Experiment 2: Batch size (~40 min)

You've been using batch_size=32. A smaller batch size means more optimizer
steps per epoch. Does that change anything for rare-class performance?

**Write your prediction BEFORE running.**

```
EXPERIMENT: Batch size 16 vs 32
──────────────────────────────
Variable changed:  ___
Held constant:     ___
Prediction:        ___
```

YOUR PREDICTION:



In [ ]:
# INTERACTIVE: Set up experiment 2 — change batch_size to 16
exp2_batch_size = ___  # set to 16

exp2 = run_experiment(
    name="sqrt-inverse bs=16",
    batch_size=exp2_batch_size,
    lr=2e-5,
    loss_fn=weighted_loss,
)

# Save incrementally
row = {k: v for k, v in exp2.items() if k not in ("history", "val_preds", "val_labels")}
pd.DataFrame([row]).to_csv(SAVE_DIR / "week2_hw_results.csv", mode="a", header=False, index=False)

**Complete the experiment template:**

```
Result:      ___
Conclusion:  ___
Meaningful?  ___
```

YOUR RESULT:



### Experiment 3: Reproducibility check (~32 min)

Same config as Experiment 1 — but with a **different random seed**.

In the lecture we said: "same config, different seed → different number."
Now you'll find out how different. If the seed-to-seed variation is larger
than the Experiment 1 vs 2 difference, then the batch size result might
just be noise.

**Fill in your prediction BEFORE running.**

```
EXPERIMENT: Reproducibility (seed=123)
──────────────────────────────────────
Variable changed:  Random seed (42 → 123)
Held constant:     Everything else (same as Experiment 1)
Prediction:        ___
```

YOUR PREDICTION:



In [ ]:
# GUIDED: Re-seed everything and re-run Experiment 1's config
import random
SEED_EXP3 = 123
random.seed(SEED_EXP3)
np.random.seed(SEED_EXP3)
torch.manual_seed(SEED_EXP3)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_EXP3)

exp3 = run_experiment(
    name="sqrt-inverse bs=32 seed=123",
    batch_size=32,
    lr=2e-5,
    loss_fn=weighted_loss,
)

row = {k: v for k, v in exp3.items() if k not in ("history", "val_preds", "val_labels")}
pd.DataFrame([row]).to_csv(SAVE_DIR / "week2_hw_results.csv", mode="a", header=False, index=False)

# Restore original seed for remaining cells
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

**Complete the template:**

```
Result:      ___
Conclusion:  ___
Meaningful?  ___
```

**Key question:** Is the difference between Experiment 1 and 3 (same config,
different seed) larger or smaller than the difference between Experiment 1
and 2 (different batch size)? What does that tell you about whether the
batch size result is real?

YOUR RESULT:



### Experiment log

In [ ]:
# GUIDED: Combine all experiments into a log
all_experiments = [exp1, exp2, exp3]
log_rows = [{k: v for k, v in e.items() if k not in ("history", "val_preds", "val_labels")}
            for e in all_experiments]
experiment_log = pd.DataFrame(log_rows)

print("=" * 70)
print("EXPERIMENT LOG")
print("=" * 70)
print(experiment_log.to_string(index=False))

---
## Part 3: Early Stopping (~45 min)

In Part 2 you trained for a fixed 3 epochs. But how do you know 3 is the
right number? The model might peak at epoch 2 and overfit by epoch 3 — or
it might still be improving and need more time.

**Early stopping** solves this: train for more epochs, monitor a metric
after each one, and stop when it stops improving.

On long-tail tasks, val loss and macro F1 can disagree — val loss may
start climbing while macro F1 is still improving. Since macro F1 is what
we actually care about, we use it as the stopping metric. But it's also
noisier than loss, so use a patience of at least 2 epochs.
---

In [ ]:
# INTERACTIVE: Implement early stopping.
# Fill in the blanks in this training loop. Use the best config from Part 2.

# INTERACTIVE: Set your best config from Part 2.
# Look at your experiment log — which experiment had the highest macro F1?
best_config_loss_fn = ___  # weighted_loss or nn.CrossEntropyLoss()
best_config_bs = ___       # 32 or 16 — whichever gave the best macro F1

MAX_EPOCHS = 10
PATIENCE = ___  # integer, e.g. 2 or 3

t_loader = DataLoader(ComplaintDataset(train_ds), batch_size=best_config_bs, shuffle=True)
v_loader = DataLoader(ComplaintDataset(val_ds), batch_size=64, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, attn_implementation="sdpa",
    id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(t_loader) * MAX_EPOCHS
warmup_steps = max(1, total_steps // 10)


def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))


scheduler = LambdaLR(optimizer, lr_lambda)
scaler = GradScaler("cuda", enabled=use_scaler)

best_macro_f1 = 0.0
patience_counter = 0
best_epoch = 0
es_history = []

print(f"Training with early stopping: max_epochs={MAX_EPOCHS}, patience={PATIENCE}")
wall_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss, n = 0.0, 0
    progress = tqdm(t_loader, desc=f"  Epoch {epoch}/{MAX_EPOCHS}", leave=True)
    for batch in progress:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        with autocast("cuda", dtype=AMP_DTYPE, enabled=(device.type == "cuda")):
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss = best_config_loss_fn(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * labels.size(0)
        n += labels.size(0)
        progress.set_postfix(loss=f"{running_loss/n:.4f}")

    # Evaluate
    model.eval()
    ep_preds, ep_labels = [], []
    with torch.no_grad():
        for batch in v_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            with autocast("cuda", dtype=AMP_DTYPE, enabled=(device.type == "cuda")):
                logits = model(input_ids=ids, attention_mask=mask).logits
            ep_preds.extend(logits.argmax(dim=-1).cpu().numpy())
            ep_labels.extend(batch["labels"].cpu().numpy())

    val_f1 = f1_score(ep_labels, ep_preds, average="macro", zero_division=0)
    val_acc = accuracy_score(ep_labels, ep_preds)
    es_history.append({"epoch": epoch, "loss": running_loss / n, "acc": val_acc, "f1": val_f1})
    print(f"  Epoch {epoch}: f1={val_f1:.4f}, acc={val_acc:.4f}")

    # INTERACTIVE: Fill in the early stopping logic
    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        best_epoch = epoch
        patience_counter = ___  # reset patience
        # Save best checkpoint
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pt")
    else:
        patience_counter += ___  # increment patience
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}. Best was epoch {best_epoch}.")
            break

train_time = time.time() - wall_start
print(f"\nBest epoch: {best_epoch} (f1={best_macro_f1:.4f}), total time: {train_time:.0f}s")

In [ ]:
# GUIDED: Load best checkpoint and get final predictions
model.load_state_dict(torch.load(SAVE_DIR / "best_model.pt", weights_only=True))
model.eval()

best_preds, best_labels = [], []
with torch.no_grad():
    for batch in v_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with autocast("cuda", dtype=AMP_DTYPE, enabled=(device.type == "cuda")):
            logits = model(input_ids=ids, attention_mask=mask).logits
        best_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        best_labels.extend(batch["labels"].cpu().numpy())

best_preds = np.array(best_preds)
best_labels = np.array(best_labels)

best_acc = accuracy_score(best_labels, best_preds)
best_f1 = f1_score(best_labels, best_preds, average="macro", zero_division=0)
print(f"Best model (epoch {best_epoch}): acc={best_acc:.4f}, f1={best_f1:.4f}")

# Clean up training model
del model, optimizer, scaler, scheduler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## Part 4: Error Analysis (~1.5 hours)
---

### Per-class F1 by frequency tier

In [ ]:
# GUIDED: Compute per-class F1 for the best model
report = classification_report(
    best_labels, best_preds,
    labels=list(range(NUM_LABELS)),
    target_names=LABEL_LIST,
    zero_division=0,
    output_dict=True,
)

class_f1 = []
for i in range(NUM_LABELS):
    name = LABEL_LIST[i]
    f1_val = report[name]["f1-score"]
    support = int(report[name]["support"])
    train_count = int(np.sum(full_labels == i))
    class_f1.append({"class": name, "f1": f1_val, "val_support": support, "train_count": train_count})

class_f1_df = pd.DataFrame(class_f1)

In [ ]:
# INTERACTIVE: Categorize classes into frequency tiers.
# Head: ≥2000 training examples. Mid: 100-1999. Tail: <100.
# Fill in the conditions.

class_f1_df["tier"] = "mid"  # default
class_f1_df.loc[class_f1_df["train_count"] >= ___, "tier"] = "head"   # fill in threshold
class_f1_df.loc[class_f1_df["train_count"] < ___, "tier"] = "tail"    # fill in threshold

tier_summary = class_f1_df.groupby("tier").agg(
    n_classes=("class", "count"),
    mean_f1=("f1", "mean"),
    zero_f1=("f1", lambda x: (x == 0).sum()),
).reindex(["head", "mid", "tail"])

print("Per-tier summary:")
print(tier_summary.to_string())

**What does this table tell you about where the model succeeds vs fails?**

YOUR ANSWER:



### Confusion matrix: where do tail-class errors land?

**Predict before you look.** When the model misclassifies a tail class,
where do you think most errors go?

(a) Head classes — rare examples get swamped by the majority
(b) Semantically similar mid-tier classes — the model knows the domain but
    can't distinguish fine-grained categories
(c) Other tail classes

YOUR PREDICTION:



In [ ]:
# GUIDED: Build confusion matrix and define class tier sets
cm = confusion_matrix(best_labels, best_preds, labels=list(range(NUM_LABELS)))

train_counts = np.bincount(full_labels, minlength=NUM_LABELS)
head_ids = set(np.where(train_counts >= 2000)[0])
mid_ids = set(np.where((train_counts >= 100) & (train_counts < 2000))[0])
tail_ids = set(np.where(train_counts < 100)[0])

print(f"Head classes: {len(head_ids)}, Mid: {len(mid_ids)}, Tail: {len(tail_ids)}")

In [ ]:
# INTERACTIVE: Where do tail-class errors land?
#
# Loop over each tail class. For each one, look at its ROW in the confusion
# matrix (cm[true_id, :]). Each off-diagonal entry is an error — the model
# predicted pred_id when the true class was true_id.
#
# Categorize each error: did the model predict a head, mid, or tail class?

tail_errors_to_head = 0
tail_errors_to_mid = 0
tail_errors_to_tail = 0
tail_errors_total = 0

for true_id in tail_ids:
    for pred_id in range(NUM_LABELS):
        if pred_id == true_id:
            continue  # correct prediction, not an error
        error_count = cm[true_id, pred_id]
        if error_count == 0:
            continue
        tail_errors_total += error_count
        # INTERACTIVE: categorize this error by checking which tier pred_id belongs to
        if ___:                  # is pred_id in the head set?
            ___ += error_count   # increment the right counter
        elif ___:                # is pred_id in the mid set?
            ___ += error_count
        else:
            ___ += error_count

print("Where do tail-class errors land?")
print(f"  → Head classes:  {tail_errors_to_head:>4} ({100*tail_errors_to_head/max(1,tail_errors_total):.1f}%)")
print(f"  → Mid classes:   {tail_errors_to_mid:>4} ({100*tail_errors_to_mid/max(1,tail_errors_total):.1f}%)")
print(f"  → Other tail:    {tail_errors_to_tail:>4} ({100*tail_errors_to_tail/max(1,tail_errors_total):.1f}%)")
print(f"  Total errors:    {tail_errors_total:>4}")

**Was your prediction right? What does this pattern tell you about the
model's failure mode — is it a frequency problem or a similarity problem?**

YOUR ANSWER:



### Hard-example inspection

In [ ]:
# GUIDED: Find the highest-margin wrong predictions
# Margin = difference between top two logits (NOT a calibrated probability).
model_for_inspection = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, attn_implementation="sdpa",
    id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
).to(device)
model_for_inspection.load_state_dict(torch.load(SAVE_DIR / "best_model.pt", weights_only=True))
model_for_inspection.eval()

all_logits = []
with torch.no_grad():
    for batch in DataLoader(ComplaintDataset(val_ds), batch_size=64, shuffle=False):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with autocast("cuda", dtype=AMP_DTYPE, enabled=(device.type == "cuda")):
            logits = model_for_inspection(input_ids=ids, attention_mask=mask).logits
        all_logits.append(logits.cpu().float().numpy())

all_logits = np.concatenate(all_logits, axis=0)
margins = np.max(all_logits, axis=1) - np.partition(all_logits, -2, axis=1)[:, -2]

# Find wrong predictions sorted by confidence
wrong_mask = best_preds != best_labels
wrong_indices = np.where(wrong_mask)[0]
wrong_margins = margins[wrong_indices]
top_wrong = wrong_indices[np.argsort(-wrong_margins)[:10]]

print("Top 10 highest-margin WRONG predictions:")
print("=" * 80)
for idx in top_wrong:
    true_name = LABEL_LIST[best_labels[idx]]
    pred_name = LABEL_LIST[best_preds[idx]]
    margin = margins[idx]
    text = val_ds[int(idx)]["text"][:200]
    print(f"\n  True:      {true_name}")
    print(f"  Predicted: {pred_name}")
    print(f"  Margin:    {margin:.2f}")
    print(f"  Text:      {text}...")

del model_for_inspection
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

**What pattern do you see in these high-margin wrong predictions?**

YOUR ANSWER:



---
## Part 5: Val Set Reliability (~20 min)
---

Your val set has 6,430 examples across 113 classes. That sounds like a lot.
But how are those examples distributed?

**Before running the next cell, predict: how many classes have 5 or fewer
validation examples?**

YOUR PREDICTION:



In [ ]:
# GUIDED: Count validation examples per class
val_labels_array = np.array(val_ds["label"])
val_counts = np.bincount(val_labels_array, minlength=NUM_LABELS)

In [ ]:
# INTERACTIVE: Plot the distribution of val examples per class.
# What do you notice about the left side of the histogram?

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(___, bins=50, color="steelblue", edgecolor="white")  # plot val_counts
ax.set_xlabel("Validation examples per class")
ax.set_ylabel("Number of classes")
ax.set_title("Distribution of Val Set Size Across 113 Classes")
plt.show()

**What do you see in the histogram?** Look at the left edge.

YOUR ANSWER:



In [ ]:
# INTERACTIVE: How many classes have very few val examples?
# Write code to count them. Don't just guess — compute it from val_counts.

classes_with_1_example = ___   # classes with exactly 1 val example
classes_with_le_5 = ___        # classes with 5 or fewer val examples

print(f"Classes with exactly 1 val example: {classes_with_1_example}")
print(f"Classes with ≤ 5 val examples:      {classes_with_le_5}")
print(f"Classes with 0 val examples:        {(val_counts == 0).sum()}")

In [ ]:
# GUIDED: Show the sparsest classes
print(f"Bottom 10 classes by val support:")
for i in np.argsort(val_counts)[:10]:
    print(f"  {LABEL_LIST[i][:55]:<55}  val={val_counts[i]}, train={train_counts[i]}")

If a class has exactly 1 validation example, its F1 is either 0.0 or 1.0
depending on whether the model gets that single example right. That's a
coin flip, not a measurement.

**Go back and look at the difference between your best and second-best
experiment in Part 2. Is that difference larger than what you'd expect
from noise on these sparse classes?**

YOUR ANSWER:



---
## Part 6: Memo Draft

Each section below corresponds to one rubric criterion. Write your
observations here while the data is fresh. You can clean up the prose
for your final submission, but do the thinking now.
---

### MEMO SECTION 1: What you tried and what happened (20 points)

> Review your experiment log. Which interventions moved macro F1,
> and by how much?

**YOUR RESPONSE:**



### MEMO SECTION 2: Class weighting (20 points)

> Did class weighting help? What did it cost? Compare your Experiment 1
> results to Delta from the lab — that's your unweighted baseline on the
> same architecture and data.

**YOUR RESPONSE:**



### MEMO SECTION 3: Batch size (20 points)

> Did batch size matter? Why might it? Compare Experiments 1 and 2
> (different batch size). Then compare Experiments 1 and 3 (same config,
> different seed). Is the batch size effect larger than seed noise?

**YOUR RESPONSE:**



### MEMO SECTION 4: Error analysis (25 points)

> What does the confusion matrix reveal about where your model fails?

**YOUR RESPONSE:**



### MEMO SECTION 5: Val set reliability (15 points)

> How trustworthy are your numbers?

**YOUR RESPONSE:**

